<a href="https://colab.research.google.com/github/Tom9229/Abhishek_Rajauria_JECRC_University_CEI_Assignments/blob/main/Week2_Abhishek_Rajauria_JECRC_University.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tesla Deliveries: end-to-end ML pipeline on sales/price data covering preprocessing, EDA, feature engineering, regression modeling, hyperparameter tuning, and time series forecasting.
Models: Linear Regression, Ridge, Lasso, Elastic Net, ARIMA, SARIMA

In [ ]:

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose

sns.set_style('whitegrid')


##PLEASE UPLOAD THE CSV FILE BEFORE EXECUTING THIS CELL


In [ ]:

df = pd.read_csv('tesla_deliveries_dataset_2015_2025.csv')
df.head()


## Data Understanding

In [ ]:

print('Shape:', df.shape)
display(df.info())
display(df.describe(include='all'))


In [ ]:

print(df.isnull().sum())
print('Duplicates:', df.duplicated().sum())


## Date Feature Creation

In [ ]:

df['Date'] = pd.to_datetime(df['Year'].astype(str)+'-'+df['Month'].astype(str)+'-01')
df = df.sort_values('Date')
df.head()


## EDA

In [ ]:

num_cols = df.select_dtypes(include=np.number).columns
df[num_cols].hist(figsize=(15,10))
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(10,6))
sns.heatmap(df[num_cols].corr(),annot=True,cmap='coolwarm')
plt.show()


In [ ]:

for col in ['Region','Model','Source_Type']:
    plt.figure(figsize=(8,4))
    sns.countplot(data=df,x=col)
    plt.xticks(rotation=45)
    plt.show()


## Feature Engineering

In [ ]:

df['Quarter']=((df['Month']-1)//3)+1
df['Half']=np.where(df['Month']<=6,1,2)
df['YearMonth']=df['Year']*100+df['Month']

df['Delivery_Efficiency']=df['Estimated_Deliveries']/df['Production_Units']
df['Price_per_km']=df['Avg_Price_USD']/df['Range_km']
df['Price_per_kWh']=df['Avg_Price_USD']/df['Battery_Capacity_kWh']
df['Charging_Density']=df['Charging_Stations']/(df['Estimated_Deliveries']+1)

for col in ['Region','Model']:
    df[f'lag1_{col}']=df.groupby(col)['Estimated_Deliveries'].shift(1)

df['rolling_mean_3']=df['Estimated_Deliveries'].rolling(3).mean()

df = df.fillna(method='bfill')


## Encoding and Preprocessing

In [ ]:

encoders = {}
for col in ['Region','Model','Source_Type']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

target = 'Estimated_Deliveries'

X = df.drop(columns=[target,'Date'])
y = df[target]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Linear Regression

In [ ]:

lr = LinearRegression()
lr.fit(X_train_scaled,y_train)

pred_lr = lr.predict(X_test_scaled)

print('MAE:',mean_absolute_error(y_test,pred_lr))
print('RMSE:',np.sqrt(mean_squared_error(y_test,pred_lr)))
print('R2:',r2_score(y_test,pred_lr))


## Ridge Regression

In [ ]:

ridge = GridSearchCV(
    Ridge(),
    {'alpha':[0.01,0.1,1,10,100]},
    cv=5,
    scoring='r2'
)
ridge.fit(X_train_scaled,y_train)
pred_ridge = ridge.predict(X_test_scaled)

print('Best Alpha:',ridge.best_params_)
print('R2:',r2_score(y_test,pred_ridge))


## Lasso Regression

In [ ]:

lasso = GridSearchCV(
    Lasso(max_iter=10000),
    {'alpha':[0.001,0.01,0.1,1,10]},
    cv=5,
    scoring='r2'
)
lasso.fit(X_train_scaled,y_train)
pred_lasso = lasso.predict(X_test_scaled)

print('Best Alpha:',lasso.best_params_)
print('R2:',r2_score(y_test,pred_lasso))


## Elastic Net

In [ ]:

enet = GridSearchCV(
    ElasticNet(max_iter=10000),
    {'alpha':[0.001,0.01,0.1,1],
     'l1_ratio':[0.2,0.5,0.8]},
    cv=5,
    scoring='r2'
)
enet.fit(X_train_scaled,y_train)
pred_enet = enet.predict(X_test_scaled)

print('Best Params:',enet.best_params_)
print('R2:',r2_score(y_test,pred_enet))


## Model Comparison

In [ ]:

results = pd.DataFrame({
'Model':['Linear','Ridge','Lasso','ElasticNet'],
'R2':[r2_score(y_test,pred_lr),
      r2_score(y_test,pred_ridge),
      r2_score(y_test,pred_lasso),
      r2_score(y_test,pred_enet)]
})
results.sort_values('R2',ascending=False)


## Diagnostics

In [ ]:

plt.figure(figsize=(6,6))
plt.scatter(y_test,pred_lr)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted')
plt.show()

residuals = y_test - pred_lr
sns.histplot(residuals,kde=True)
plt.title('Residual Distribution')
plt.show()


##**HYPERPARAMETER TUNING**

##RIDGE REGRESSION

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge

ridge_params = {
    'alpha': [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}

ridge_grid = GridSearchCV(
    Ridge(),
    param_grid=ridge_params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

ridge_grid.fit(X_train_scaled, y_train)

best_ridge = ridge_grid.best_estimator_

print("Best Parameters:", ridge_grid.best_params_)
print("Best CV Score:", ridge_grid.best_score_)

ridge_pred = best_ridge.predict(X_test_scaled)

##LASSO REGRESSION

In [ ]:
from sklearn.linear_model import Lasso

lasso_params = {
    'alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10]
}

lasso_grid = GridSearchCV(
    Lasso(max_iter=10000),
    param_grid=lasso_params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

lasso_grid.fit(X_train_scaled, y_train)

best_lasso = lasso_grid.best_estimator_

print("Best Parameters:", lasso_grid.best_params_)
print("Best CV Score:", lasso_grid.best_score_)

lasso_pred = best_lasso.predict(X_test_scaled)

##ELASTIC NET

In [ ]:
from sklearn.linear_model import ElasticNet

elastic_params = {
    'alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

elastic_grid = GridSearchCV(
    ElasticNet(max_iter=10000),
    param_grid=elastic_params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

elastic_grid.fit(X_train_scaled, y_train)

best_elastic = elastic_grid.best_estimator_

print("Best Parameters:", elastic_grid.best_params_)
print("Best CV Score:", elastic_grid.best_score_)

elastic_pred = best_elastic.predict(X_test_scaled)

##CROSS VALIDATION SCORE

In [ ]:
from sklearn.model_selection import cross_val_score

models = {
    'Linear Regression': LinearRegression(),
    'Ridge': best_ridge,
    'Lasso': best_lasso,
    'Elastic Net': best_elastic
}

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train_scaled,
        y_train,
        cv=5,
        scoring='r2'
    )

    print(f"{name}")
    print(f"Mean R²: {scores.mean():.4f}")
    print(f"Std R²:  {scores.std():.4f}")
    print("-"*40)

### Comparison of Models Before and After Hyperparameter Tuning

In [ ]:
tuned_ridge_pred = best_ridge.predict(X_test_scaled)
tuned_lasso_pred = best_lasso.predict(X_test_scaled)
tuned_elastic_pred = best_elastic.predict(X_test_scaled)

comparison_results = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge (Before Tuning)', 'Lasso (Before Tuning)', 'ElasticNet (Before Tuning)',
              'Ridge (After Tuning)', 'Lasso (After Tuning)', 'ElasticNet (After Tuning)']
})

# Get R2 scores from the initial results DataFrame
initial_r2_lr = results[results['Model'] == 'Linear']['R2'].values[0]
initial_r2_ridge = results[results['Model'] == 'Ridge']['R2'].values[0]
initial_r2_lasso = results[results['Model'] == 'Lasso']['R2'].values[0]
initial_r2_enet = results[results['Model'] == 'ElasticNet']['R2'].values[0]

comparison_results['R2 Score'] = [
    initial_r2_lr,
    initial_r2_ridge,
    initial_r2_lasso,
    initial_r2_enet,
    r2_score(y_test, tuned_ridge_pred),
    r2_score(y_test, tuned_lasso_pred),
    r2_score(y_test, tuned_elastic_pred)
]

display(comparison_results.sort_values(by='R2 Score', ascending=False))

In [ ]:
plt.figure(figsize=(12, 7))
sns.barplot(x='Model', y='R2 Score', data=comparison_results.sort_values(by='R2 Score', ascending=False), palette='viridis')
plt.title('R2 Score Comparison: Before and After Hyperparameter Tuning')
plt.xlabel('Model')
plt.ylabel('R2 Score')
plt.xticks(rotation=45, ha='right')
plt.ylim(0.998, 1.0) # Set a relevant y-axis limit for better visibility of differences
plt.tight_layout()
plt.show()

## Time Series Preparation

In [ ]:

ts = df.groupby('Date')['Estimated_Deliveries'].sum()
ts.plot(figsize=(12,5),title='Monthly Deliveries')
plt.show()


In [ ]:

decomp = seasonal_decompose(ts, model='additive', period=12)
decomp.plot()
plt.show()


## ADF Test

In [ ]:

result = adfuller(ts)
print('ADF Statistic:',result[0])
print('p-value:',result[1])


In [ ]:

plot_acf(ts,lags=24)
plt.show()

plot_pacf(ts,lags=24)
plt.show()


## ARIMA Forecasting

In [ ]:

train = ts[:-12]
test = ts[-12:]

arima = ARIMA(train, order=(1,1,1))
arima_fit = arima.fit()

forecast_arima = arima_fit.forecast(steps=12)

print('ARIMA RMSE:', np.sqrt(mean_squared_error(test, forecast_arima)))


## SARIMA Forecasting

In [ ]:

sarima = SARIMAX(train,
                 order=(1,1,1),
                 seasonal_order=(1,1,1,12))
sarima_fit = sarima.fit(disp=False)

forecast_sarima = sarima_fit.forecast(steps=12)

print('SARIMA RMSE:', np.sqrt(mean_squared_error(test, forecast_sarima)))


In [ ]:

plt.figure(figsize=(12,5))
plt.plot(train.index,train,label='Train')
plt.plot(test.index,test,label='Actual')
plt.plot(test.index,forecast_arima,label='ARIMA Forecast')
plt.plot(test.index,forecast_sarima,label='SARIMA Forecast')
plt.legend()
plt.show()
